<h1> Customer Retention and Cohort Analysis <h1>

<h3> Check Installs / Make Imports <h3>

In [ ]:
#Double Check Installs
%pip install -q duckdb pandas pyarrow

In [1]:
#make import
from pathlib import Path
import duckdb

WindowsPath('C:/Users/Emmet/PycharmProjects/codespaces-jupyter/data/raw/online_retail_II.csv')

<h3> Align Data Folders <h3>

In [ ]:
#folder structure
ROOT = Path.cwd()  #repo root
RAW = ROOT / "data" / "raw"
PROS = ROOT / "data" / "processed"
PROS.mkdir(parents=True, exist_ok=True)

con = duckdb.connect(str(ROOT / "retail.duckdb")) #connect to a file-backed DB so it persists

#point to your CSV (adjust if needed)
csv_path = RAW / "online_retail_II.csv"
assert csv_path.exists(), f"Missing file: {csv_path}"
csv_path

In [2]:
#preview 5 rows to make sure it's all there
con.execute(f"""
    SELECT * FROM read_csv_auto('{csv_path}', HEADER=TRUE)
    LIMIT 5
""").df()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
#check number of rows
con.execute(f"""
    SELECT COUNT(*) FROM read_csv_auto('{csv_path}', HEADER=TRUE)
    LIMIT 5
""").df()

,count_star()
0,1067371


<h3> Data Preparation <h3>

In [4]:
#create/replace base table for raw data
con.execute(f"""
    CREATE OR REPLACE TABLE base_table AS
    SELECT *
    FROM read_csv_auto('{csv_path}', HEADER=TRUE)
""")

#check table got made with correct number of rows
con.execute("""SELECT COUNT(*) AS rows FROM base_table""").df()

,rows
0,1067371


In [5]:
#check for min and max dates
con.execute("SELECT MIN(InvoiceDate), MAX(InvoiceDate) FROM base_table").df()

,min(InvoiceDate),max(InvoiceDate)
0,2009-12-01 07:45:00,2011-12-09 12:50:00


In [6]:
#find number of columns where customer ID isn't present
con.execute('SELECT COUNT(*) AS null_customers FROM base_table WHERE "Customer ID" IS NULL').df()


,null_customers
0,243007


In [7]:
#create the main table we're going to be working with
#remove columns with no customer ID. Make sure there is an amount purchased and a price on it.
#change table columns to more programming-ish names.

con.execute("""
    CREATE OR REPLACE TABLE real_table AS
    SELECT
      CAST("Customer ID" AS BIGINT)          AS customer_id,
      CAST(Quantity AS INTEGER)              AS quantity,
      CAST(Price AS DOUBLE)                  AS unit_price,
      Quantity * Price                       AS revenue,
      InvoiceDate                            AS invoice_ts,
      DATE_TRUNC('month', InvoiceDate)       AS invoice_month,
      Country                                AS country
    FROM base_table
    WHERE "Customer ID" IS NOT NULL
      AND Quantity > 0
      AND Price > 0
""")

In [8]:
#row and revenue count
con.execute("""SELECT COUNT(*) AS rows, SUM(revenue) AS revenue FROM real_table""").df()

,rows,revenue
0,805549,1.774343e+07


In [9]:
#preview new table
con.execute(f"""
    SELECT * FROM real_table
    LIMIT 5
""").df()

,customer_id,quantity,unit_price,revenue,invoice_ts,invoice_month,country
0,13085,12,6.95,83.4,2009-12-01 07:45:00,2009-12-01,United Kingdom
1,13085,12,6.75,81.0,2009-12-01 07:45:00,2009-12-01,United Kingdom
2,13085,12,6.75,81.0,2009-12-01 07:45:00,2009-12-01,United Kingdom
3,13085,48,2.10,100.8,2009-12-01 07:45:00,2009-12-01,United Kingdom
4,13085,24,1.25,30.0,2009-12-01 07:45:00,2009-12-01,United Kingdom


In [10]:
#time still correct?
con.execute("""
    SELECT MIN(invoice_ts) AS min_ts, MAX(invoice_ts) AS max_ts /*  */
    FROM real_table
""").df()

,min_ts,max_ts
0,2009-12-01 07:45:00,2011-12-09 12:50:00


Retention Metrics Table

In [11]:
#create a table for the cohort retention metrics
#looking for a table of customers and their cohort_month(first purchase), current activity month, cohort index (how many months since first purchase), activity flag (are they active in the month).
#will help us compute retention

#first_order will be (customer_id, cohort_month) or Customers x first month they purchased
#activity will be (customer_id, invoice_month, month_revenue) or one row per customer per month with the sum of revenue that month.
#cohorted will be (customer_id, cohort_month, invoice_month, cohort_index, active_flag)

con.execute("""
    CREATE OR REPLACE TABLE cohort_ret_metrics AS
    WITH first_order AS (                                       /* create first_order as CTE */
      SELECT customer_id, MIN(invoice_month) AS cohort_month
      FROM real_table
      GROUP BY customer_id
    ),
    activity AS (                                               /* create activity as CTE */
      SELECT customer_id, invoice_month, SUM(revenue) AS month_revenue
      FROM real_table
      GROUP BY customer_id, invoice_month
    ),
    cohorted AS (                             /* create cohorted as CTE from activity and first_order */
      SELECT
        a.customer_id,
        f.cohort_month,
        a.invoice_month,
        DATE_DIFF('month', f.cohort_month, a.invoice_month) AS cohort_index,
        CASE WHEN a.month_revenue > 0 THEN 1 ELSE 0 END     AS active_flag
      FROM activity a
      JOIN first_order f USING (customer_id)
    ),
    cohort_sizes AS (
      SELECT cohort_month, COUNT(DISTINCT customer_id) AS cohort_size
      FROM cohorted
      WHERE cohort_index = 0
      GROUP BY cohort_month
    )
    SELECT
      c.cohort_month,
      c.cohort_index,
      COUNT(DISTINCT CASE WHEN c.active_flag = 1 THEN c.customer_id END) AS active_customers,
      s.cohort_size,
      ROUND(100.0 * COUNT(DISTINCT CASE WHEN c.active_flag = 1 THEN c.customer_id END)::DOUBLE
            / s.cohort_size, 1) AS retention_pct
    FROM cohorted c
    JOIN cohort_sizes s USING (cohort_month)
    GROUP BY 1,2,4
    ORDER BY 1,2
""")

# end with cohort_ret_metrics, which has one row per cohort_month x cohort_index


In [12]:
# Cohort month (index 0) must always be 100% retained
con.execute("""
  SELECT SUM(CASE WHEN cohort_index=0 AND retention_pct=100 THEN 1 ELSE 0 END) AS ok_rows,
         COUNT(*) FILTER (WHERE cohort_index=0) AS total_zero_rows
  FROM cohort_ret_metrics
""").df()

,ok_rows,total_zero_rows
0,25.0,25


In [13]:
#check a couple rows to make sure it's all going smooth
con.execute("SELECT * FROM cohort_ret_metrics ORDER BY cohort_month, cohort_index LIMIT 10").df()


,cohort_month,cohort_index,active_customers,cohort_size,retention_pct
0,2009-12-01,0,955,955,100.0
1,2009-12-01,1,337,955,35.3
2,2009-12-01,2,319,955,33.4
3,2009-12-01,3,406,955,42.5
4,2009-12-01,4,363,955,38.0
5,2009-12-01,5,343,955,35.9
6,2009-12-01,6,360,955,37.7
7,2009-12-01,7,327,955,34.2
8,2009-12-01,8,321,955,33.6
9,2009-12-01,9,346,955,36.2


In [14]:
#KPI series for dashboard

con.execute("""
    CREATE OR REPLACE TABLE mart_kpi_timeseries AS
    SELECT
      invoice_month,
      COUNT(DISTINCT customer_id) AS active_users,
      SUM(revenue)                AS revenue
    FROM real_table
    GROUP BY 1
    ORDER BY 1
""")

In [15]:
# Check that all cohort_index=0 rows are exactly 100% retained.
con.execute("""
SELECT
  SUM(CASE WHEN cohort_index=0 AND retention_pct=100 THEN 1 ELSE 0 END) AS ok_rows,
  COUNT(*) FILTER (WHERE cohort_index=0) AS total_zero_rows
FROM cohort_ret_metrics
""").df()

,ok_rows,total_zero_rows
0,25.0,25


In [20]:
#check last 6 months of cohorts to make sure data is still making sense

con.execute("""
SELECT
  cohort_month,
  cohort_index,
  cohort_size,
  active_customers,
  retention_pct
FROM cohort_ret_metrics
WHERE cohort_month >= (SELECT MAX(cohort_month) - INTERVAL 6 MONTH FROM cohort_ret_metrics )
ORDER BY cohort_month DESC, cohort_index
LIMIT 60
""").df()

,cohort_month,cohort_index,cohort_size,active_customers,retention_pct
0,2011-12-01,0,28,28,100.0
1,2011-11-01,0,191,191,100.0
2,2011-11-01,1,191,27,14.1
3,2011-10-01,0,221,221,100.0
4,2011-10-01,1,221,71,32.1
5,2011-10-01,2,221,35,15.8
6,2011-09-01,0,189,189,100.0
7,2011-09-01,1,189,51,27.0
8,2011-09-01,2,189,71,37.6
9,2011-09-01,3,189,28,14.8


<h3> Export Data for Tableau Upload <h3>

In [19]:
#Export tables

PROC = Path("data/processed")                 # place to store stable CSVs
PROC.mkdir(parents=True, exist_ok=True)       # create the folder if missing

# Export the retention metrics (heatmap + curves source)
con.execute(f"""
  COPY (SELECT * FROM cohort_ret_metrics)
  TO '{(PROC / "retention_monthly.csv")}'
  WITH (HEADER, DELIMITER ',')
""")

# Export the KPI time series (header tiles + context lines)
con.execute(f"""
  COPY (SELECT * FROM mart_kpi_timeseries)
  TO '{(PROC / "kpi_timeseries.csv")}'
  WITH (HEADER, DELIMITER ',')
""")

In [21]:
#quick question - were any cohorts greater than 60% retention? Nope

con.execute("""
SELECT cohort_month, cohort_index, retention_pct
FROM cohort_ret_metrics
WHERE retention_pct > 60
ORDER BY retention_pct DESC
""").df()

,cohort_month,cohort_index,retention_pct
0,2009-12-01,0,100.0
1,2010-01-01,0,100.0
2,2010-02-01,0,100.0
3,2010-03-01,0,100.0
4,2010-04-01,0,100.0
5,2010-05-01,0,100.0
6,2010-06-01,0,100.0
7,2010-07-01,0,100.0
8,2010-08-01,0,100.0
9,2010-09-01,0,100.0


Retention Metrics and KPI timeseries are now exported for use in Tableau. Please see deliverables file for a summary of results and dashboard images. Link to dashboard: